# Week 1: Data Validation and Exploratory Data Analysis

## Project Objective

This project focuses on multi-touch marketing attribution. The aim is to understand how different marketing touchpoints contribute to customer purchases.

The available datasets include web events, transactions, customers, products, and campaign metadata.

The current phase focuses on:
- Dataset validation
- Customer journey analysis
- Funnel analysis
- First-touch attribution preparation
- Last-touch attribution preparation
- Linear attribution preparation

## Data Limitation

The shared datasets do not currently include ad spend, campaign cost, budget, CPC, or CPM data.

Therefore, true ROAS, CAC, CPC, and ROI cannot be calculated directly at this stage. These metrics will require an additional ad_spend dataset or a synthetic ad_spend table later.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
DATA_DIR = Path("../data")

events_path = DATA_DIR / "events.csv"
transactions_path = DATA_DIR / "transactions.csv"
customers_path = DATA_DIR / "customers.csv"
products_path = DATA_DIR / "products.csv"
campaigns_path = DATA_DIR / "campaigns.csv"

required_files = {
    "events": events_path,
    "transactions": transactions_path,
    "customers": customers_path,
    "products": products_path,
    "campaigns": campaigns_path
}

for name, path in required_files.items():
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'} - {path}")

events: FOUND - ..\data\events.csv
transactions: FOUND - ..\data\transactions.csv
customers: FOUND - ..\data\customers.csv
products: FOUND - ..\data\products.csv
campaigns: FOUND - ..\data\campaigns.csv


In [5]:
events = pd.read_csv(events_path, low_memory=False)
transactions = pd.read_csv(transactions_path, low_memory=False)
customers = pd.read_csv(customers_path, low_memory=False)
products = pd.read_csv(products_path, low_memory=False)
campaigns = pd.read_csv(campaigns_path, low_memory=False)

print("Data loaded successfully.")

Data loaded successfully.


In [6]:
datasets = {
    "events": events,
    "transactions": transactions,
    "customers": customers,
    "products": products,
    "campaigns": campaigns
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows and {df.shape[1]} columns")

events: 2,000,000 rows and 12 columns
transactions: 103,127 rows and 9 columns
customers: 100,000 rows and 7 columns
products: 2,000 rows and 6 columns
campaigns: 50 rows and 7 columns


In [7]:
for name, df in datasets.items():
    print(f"\n{name.upper()} COLUMNS")
    print(df.columns.tolist())


EVENTS COLUMNS
['event_id', 'timestamp', 'customer_id', 'session_id', 'event_type', 'product_id', 'device_type', 'traffic_source', 'campaign_id', 'page_category', 'session_duration_sec', 'experiment_group']

TRANSACTIONS COLUMNS
['transaction_id', 'timestamp', 'customer_id', 'product_id', 'quantity', 'discount_applied', 'gross_revenue', 'campaign_id', 'refund_flag']

CUSTOMERS COLUMNS
['customer_id', 'signup_date', 'country', 'age', 'gender', 'loyalty_tier', 'acquisition_channel']

PRODUCTS COLUMNS
['product_id', 'category', 'brand', 'base_price', 'launch_date', 'is_premium']

CAMPAIGNS COLUMNS
['campaign_id', 'channel', 'objective', 'start_date', 'end_date', 'target_segment', 'expected_uplift']


In [8]:
def missing_value_report(df):
    report = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percent": (df.isna().sum().values / len(df) * 100).round(2)
    })
    return report.sort_values("missing_count", ascending=False)

for name, df in datasets.items():
    print(f"\nMissing values in {name}:")
    display(missing_value_report(df))


Missing values in events:


,column,missing_count,missing_percent
5,product_id,200371,10.02
6,device_type,40300,2.02
0,event_id,0,0.00
1,timestamp,0,0.00
2,customer_id,0,0.00
3,session_id,0,0.00
4,event_type,0,0.00
7,traffic_source,0,0.00
8,campaign_id,0,0.00
9,page_category,0,0.00



Missing values in transactions:


,column,missing_count,missing_percent
3,product_id,10449,10.13
6,gross_revenue,10449,10.13
0,transaction_id,0,0.00
1,timestamp,0,0.00
2,customer_id,0,0.00
4,quantity,0,0.00
5,discount_applied,0,0.00
7,campaign_id,0,0.00
8,refund_flag,0,0.00



Missing values in customers:


,column,missing_count,missing_percent
0,customer_id,0,0.0
1,signup_date,0,0.0
2,country,0,0.0
3,age,0,0.0
4,gender,0,0.0
5,loyalty_tier,0,0.0
6,acquisition_channel,0,0.0



Missing values in products:


,column,missing_count,missing_percent
0,product_id,0,0.0
1,category,0,0.0
2,brand,0,0.0
3,base_price,0,0.0
4,launch_date,0,0.0
5,is_premium,0,0.0



Missing values in campaigns:


,column,missing_count,missing_percent
0,campaign_id,0,0.0
1,channel,0,0.0
2,objective,0,0.0
3,start_date,0,0.0
4,end_date,0,0.0
5,target_segment,0,0.0
6,expected_uplift,0,0.0


In [9]:
spend_keywords = ["spend", "cost", "budget", "cpc", "cpm", "roas", "cac"]

for name, df in datasets.items():
    spend_columns = [
        col for col in df.columns
        if any(keyword in col.lower() for keyword in spend_keywords)
    ]
    print(f"{name}: {spend_columns}")

events: []
transactions: []
customers: []
products: []
campaigns: []


## Ad Spend Data Availability Check

The current datasets include events, transactions, customers, products, and campaigns.

After checking all column names, no spend, cost, budget, CPC, CPM, ROAS, or CAC-related column was found.

Therefore, true ROAS and CAC cannot be calculated directly from the current data. The current analysis will focus on customer journey analysis, funnel metrics, attribution models, and attributed revenue. A synthetic ad spend table can be created later if approved by the team.

## Missing Value Analysis

This section checks missing values in each dataset. Missing values are important because they may affect joins, revenue calculation, attribution logic, and dashboard accuracy.

In [10]:
def missing_value_report(df):
    report = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percent": (df.isna().sum().values / len(df) * 100).round(2)
    })
    return report.sort_values("missing_count", ascending=False)

for name, df in datasets.items():
    print(f"\nMissing values in {name}:")
    display(missing_value_report(df))


Missing values in events:


,column,missing_count,missing_percent
5,product_id,200371,10.02
6,device_type,40300,2.02
0,event_id,0,0.00
1,timestamp,0,0.00
2,customer_id,0,0.00
3,session_id,0,0.00
4,event_type,0,0.00
7,traffic_source,0,0.00
8,campaign_id,0,0.00
9,page_category,0,0.00



Missing values in transactions:


,column,missing_count,missing_percent
3,product_id,10449,10.13
6,gross_revenue,10449,10.13
0,transaction_id,0,0.00
1,timestamp,0,0.00
2,customer_id,0,0.00
4,quantity,0,0.00
5,discount_applied,0,0.00
7,campaign_id,0,0.00
8,refund_flag,0,0.00



Missing values in customers:


,column,missing_count,missing_percent
0,customer_id,0,0.0
1,signup_date,0,0.0
2,country,0,0.0
3,age,0,0.0
4,gender,0,0.0
5,loyalty_tier,0,0.0
6,acquisition_channel,0,0.0



Missing values in products:


,column,missing_count,missing_percent
0,product_id,0,0.0
1,category,0,0.0
2,brand,0,0.0
3,base_price,0,0.0
4,launch_date,0,0.0
5,is_premium,0,0.0



Missing values in campaigns:


,column,missing_count,missing_percent
0,campaign_id,0,0.0
1,channel,0,0.0
2,objective,0,0.0
3,start_date,0,0.0
4,end_date,0,0.0
5,target_segment,0,0.0
6,expected_uplift,0,0.0


## Missing Value Findings

The missing value report shows that some fields contain missing values.

In the events dataset, missing product_id values may occur because not every web event is related to a specific product. For example, home page views, checkout page visits, or bounce events may not always have a product_id.

In the transactions dataset, missing product_id and gross_revenue values are more important because they affect revenue and product-level analysis. Rows with missing gross_revenue should not be used for revenue attribution calculations.

For attribution modeling, only valid non-refunded transactions with available gross_revenue will be used.

## Timestamp and Date Conversion

Attribution depends on the correct order of customer events. Therefore, timestamp and date columns must be converted into proper datetime format.

In [12]:
events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"], errors="coerce")
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
products["launch_date"] = pd.to_datetime(products["launch_date"], errors="coerce")
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"], errors="coerce")
campaigns["end_date"] = pd.to_datetime(campaigns["end_date"], errors="coerce")

print("Date conversion completed.")

Date conversion completed.


In [13]:
date_columns = {
    "events.timestamp": events["timestamp"],
    "transactions.timestamp": transactions["timestamp"],
    "customers.signup_date": customers["signup_date"],
    "products.launch_date": products["launch_date"],
    "campaigns.start_date": campaigns["start_date"],
    "campaigns.end_date": campaigns["end_date"]
}

for label, series in date_columns.items():
    print(f"\n{label}")
    print(f"Missing or invalid dates: {series.isna().sum():,}")
    print(f"Minimum date: {series.min()}")
    print(f"Maximum date: {series.max()}")


events.timestamp
Missing or invalid dates: 0
Minimum date: 2021-01-01 00:01:28
Maximum date: 2023-12-31 23:57:50

transactions.timestamp
Missing or invalid dates: 0
Minimum date: 2021-01-01 00:12:50
Maximum date: 2023-12-31 22:37:32

customers.signup_date
Missing or invalid dates: 0
Minimum date: 2021-01-01 00:00:00
Maximum date: 2023-12-31 00:00:00

products.launch_date
Missing or invalid dates: 0
Minimum date: 2021-01-01 00:00:00
Maximum date: 2023-12-31 00:00:00

campaigns.start_date
Missing or invalid dates: 0
Minimum date: 2021-01-20 00:00:00
Maximum date: 2023-11-04 00:00:00

campaigns.end_date
Missing or invalid dates: 0
Minimum date: 2021-02-21 00:00:00
Maximum date: 2024-01-06 00:00:00


## Traffic Source Cleaning

The traffic_source column may contain inconsistent capitalization such as Organic and ORGANIC. This section standardizes traffic source values for accurate grouping.

In [14]:
events["traffic_source_clean"] = (
    events["traffic_source"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
    .str.title()
)

print("Original traffic source values:")
print(events["traffic_source"].value_counts())

print("\nCleaned traffic source values:")
print(events["traffic_source_clean"].value_counts())

Original traffic source values:
Organic        776758
Paid Search    387657
Social         291194
Email          290651
Direct         193870
ORGANIC         23731
PAID SEARCH     12181
SOCIAL           9077
EMAIL            8989
DIRECT           5892
Name: traffic_source, dtype: int64

Cleaned traffic source values:
Organic        800489
Paid Search    399838
Social         300271
Email          299640
Direct         199762
Name: traffic_source_clean, dtype: int64


## campaign_id = 0 Check

The campaigns table contains official campaign IDs. However, events and transactions may contain campaign_id = 0. This is likely organic, direct, no campaign, or unattributed activity.

In [15]:
print("Events with campaign_id = 0:", (events["campaign_id"] == 0).sum())
print("Transactions with campaign_id = 0:", (transactions["campaign_id"] == 0).sum())
print("Campaign ID range in campaigns table:", campaigns["campaign_id"].min(), "to", campaigns["campaign_id"].max())

Events with campaign_id = 0: 1000251
Transactions with campaign_id = 0: 20955
Campaign ID range in campaigns table: 1 to 50


campaign_id = 0 will be treated as "No Campaign / Organic / Direct / Unattributed" instead of being treated as an error. This helps preserve unattributed customer activity during analysis.